# 🤖 Selenium × AI 프롬프트 완전 활용법
## — 코드를 외우지 말고, AI와 협업으로 동적 사이트 크롤링하기

> **이 노트북의 핵심 메시지**
> Selenium 문법을 외울 필요 없습니다.
> **"어떤 데이터가 필요한지 AI에게 잘 설명하는 능력"** 이 더 중요합니다.

---

## 이 노트북에서 배울 것

| Part | 내용 |
|------|------|
| 1 | 언제 Selenium이 필요한가? (판별법) |
| 2 | AI에게 Selenium 코드 요청하는 법 |
| 3 | 좋은 프롬프트 vs 나쁜 프롬프트 비교 |
| 4 | 상황별 프롬프트 템플릿 5종 |
| 5 | AI가 준 코드 실행하고 검증하는 법 |
| 6 | 에러가 났을 때 AI에게 디버깅 요청하는 법 |
| 7 | 실전 사례: 무신사 스토어 크롤링 |

---
## Part 1. 언제 Selenium이 필요한가?

### 판별 방법: Ctrl+U 테스트 (30초)

```
크롬에서 수집하고 싶은 페이지 열기
→ Ctrl+U (페이지 소스 보기)
→ Ctrl+F → 수집하고 싶은 텍스트 검색

텍스트가 보임  → 정적 사이트 → BeautifulSoup (앞 노트북)
텍스트가 없음  → 동적 사이트 → Selenium 필요 ← 이 노트북!
```

### Selenium이 필요한 대표적인 상황

| 상황 | 예시 사이트 |
|------|------------|
| JavaScript로 데이터를 그려주는 경우 | 무신사, 쿠팡, 네이버 쇼핑 |
| 스크롤 해야 데이터가 로드되는 경우 (무한스크롤) | 인스타그램, 트위터 스타일 |
| 버튼 클릭 후 결과가 바뀌는 경우 | 필터, 정렬 버튼 |
| 로그인 후에만 볼 수 있는 경우 | 마이페이지, 주문내역 |

In [ ]:
# BeautifulSoup이 실패하는 순간 — 직접 확인
import requests
from bs4 import BeautifulSoup

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'
}

# 정적 버전 (BeautifulSoup으로 OK)
r_static = requests.get('http://quotes.toscrape.com/')
soup_static = BeautifulSoup(r_static.text, 'html.parser')
print(f"정적 버전:    {len(soup_static.select('.quote'))}개 명언 발견 ✅")

# JS 렌더링 버전 (BeautifulSoup으로 0개)
r_js = requests.get('http://quotes.toscrape.com/js/')
soup_js = BeautifulSoup(r_js.text, 'html.parser')
print(f"JS 렌더링 버전: {len(soup_js.select('.quote'))}개 명언 발견 ← 0개!")

print()
print("→ 같은 '.quote' 선택자인데 JS 버전은 0개가 나옵니다.")
print("  이런 사이트에는 Selenium이 필요해요.")

---
## Part 2. AI에게 Selenium 코드 요청하는 법

### 왜 AI를 쓰나?

Selenium 코드는 길고 복잡합니다.

```python
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time, pandas as pd
# ... 이게 다 뭔지 외울 필요 없어요!
```

하지만 이 코드를 **이해하는 것**은 중요합니다:
- AI가 준 코드를 수정할 수 있어야 함
- 에러가 났을 때 어디가 문제인지 판단할 수 있어야 함
- 결과가 제대로 나왔는지 확인할 수 있어야 함

---

### AI에게 요청할 때 필요한 정보

```
1. 사이트 URL
2. 수집하고 싶은 데이터 항목들
3. 몇 페이지까지 수집할지
4. 저장 형식 (CSV 등)
5. 특이사항 (무한스크롤인지, 팝업이 있는지 등)
```

---
## Part 3. 좋은 프롬프트 vs 나쁜 프롬프트

### ❌ 나쁜 프롬프트

```
"무신사에서 상품 크롤링해줘"
```

**문제점:**
- 어떤 상품인지 모름
- 어떤 데이터를 원하는지 모름
- 페이지 수를 모름
- 결과를 어떻게 쓸지 모름

→ AI가 너무 일반적인 코드를 줘서 실제로 안 돌아갈 가능성 높음

---

### ✅ 좋은 프롬프트 구조

```
[상황 설명]
[수집 목표]
[기술 조건]
[출력 형식]
```

---

### 좋은 프롬프트 예시 (무신사)

```
무신사 스토어 (https://www.musinsa.com/main/musinsa/recommend?gf=A)
에서 추천 상품 목록을 Selenium으로 크롤링하는 Python 코드를 작성해줘.

[수집할 항목]
- 상품명
- 브랜드명
- 가격 (정가 / 할인가)
- 할인율
- 좋아요 수 (있으면)

[조건]
- selenium과 webdriver-manager 사용
- ChromeOptions() 기본 설정 포함
- WebDriverWait(driver, 10) 사용 (time.sleep 최소화)
- try-finally로 driver.quit() 반드시 실행
- 스크롤이 필요하면 스크롤 처리 포함
- 팝업이 있을 수 있으니 팝업 닫기 처리 포함
- 결과를 pandas DataFrame으로 저장
- CSV 저장 시 encoding='utf-8-sig' 사용 (한글 엑셀 호환)
- 주석을 한국어로 상세히 달아줘
- Jupyter Notebook에서 실행할 수 있게 셀 단위로 구분해줘
```

---

### 프롬프트 핵심 4가지

```
1. URL을 정확히 넣는다
2. 원하는 항목을 구체적으로 나열한다
3. 기술 조건을 명시한다 (WebDriverWait, try-finally 등)
4. 출력 형식을 지정한다 (DataFrame, CSV, 셀 구분 등)
```

---
## Part 4. 상황별 프롬프트 템플릿 5종

아래 템플릿에서 `[대괄호]` 부분만 바꾸면 됩니다.

---

### 템플릿 1: 기본 JS 렌더링 사이트

```
[사이트명] ([URL])에서 BeautifulSoup으로 크롤링하려고 했는데
Ctrl+U 소스에 데이터가 없어서 Selenium이 필요합니다.

[수집할 항목]
- [항목1]: [설명 또는 위치]
- [항목2]: [설명 또는 위치]
- [항목3]: [설명 또는 위치]

[조건]
- selenium과 webdriver-manager 사용
- ChromeOptions() 기본 설정 (disable-gpu, no-sandbox, window-size 포함)
- WebDriverWait(driver, 10) 사용
- try-finally로 driver.quit() 보장
- 결과를 pandas DataFrame으로 저장 후 CSV (utf-8-sig)
- 주석 한국어로 상세히
- Jupyter 셀 단위로 구분
```

---

### 템플릿 2: 무한스크롤 사이트

```
[사이트명] ([URL])은 무한스크롤 방식입니다.
스크롤을 내리면 계속 데이터가 로드됩니다.

[수집할 항목]
- [항목들]

[스크롤 조건]
- 최대 [N]번 스크롤
- 스크롤 전후 아이템 수를 비교해서 변화가 없으면 자동 종료
- 각 스크롤 후 2초 대기

[공통 조건]
- selenium + webdriver-manager
- WebDriverWait 사용, try-finally 보장
- DataFrame → CSV (utf-8-sig)
- 주석 한국어, Jupyter 셀 구분
```

---

### 템플릿 3: 버튼 클릭 후 수집

```
[사이트명] ([URL])에서 [버튼명] 버튼을 클릭하면
[어떤 데이터]가 나옵니다. 이걸 수집하고 싶습니다.

[버튼 정보]
- 버튼 텍스트 또는 위치: [설명]
- 버튼 클릭 후 로드 대기 필요

[수집할 항목]
- [항목들]

[공통 조건]
- WebDriverWait로 버튼 클릭 가능 상태 대기
- ElementClickInterceptedException 처리 포함
- try-finally, DataFrame → CSV
- 주석 한국어, Jupyter 셀 구분
```

---

### 템플릿 4: 팝업 처리 포함

```
[사이트명] ([URL])에 접속하면 팝업이 뜹니다.
팝업을 닫고 본 데이터를 수집해주세요.

[팝업 정보]
- 팝업이 항상 뜨는지, 가끔 뜨는지: [설명]
- 닫기 버튼의 텍스트 또는 위치: [설명 또는 모름]

[수집할 항목]
- [항목들]

[조건]
- 팝업 처리는 try-except로 (없으면 그냥 넘어가게)
- WebDriverWait, try-finally, DataFrame → CSV
- 주석 한국어, Jupyter 셀 구분
```

---

### 템플릿 5: 여러 페이지 (다음 버튼 클릭)

```
[사이트명] ([URL])에서 "다음" 버튼을 클릭해서
여러 페이지의 데이터를 수집하고 싶습니다.

[페이지 구조]
- 최대 [N]페이지까지 수집
- 다음 버튼이 없으면 (마지막 페이지) 자동 종료

[수집할 항목]
- [항목들]

[조건]
- 각 페이지 이동 후 데이터 로드 대기 (WebDriverWait)
- 페이지별로 수집 진행 상황 출력
- try-finally, DataFrame → CSV
- 주석 한국어, Jupyter 셀 구분
```

---
## Part 5. AI가 준 코드 실행하고 검증하는 법

### 실행 순서 (중요!)

```
1. 설치 확인 셀 실행 (패키지가 있는지)
2. import 셀 실행
3. 브라우저 옵션 셀 실행 (Chrome 창이 열림)
4. URL 이동 셀 실행 (실제 페이지로 이동)
   ← 이때 브라우저 창을 직접 보면서 제대로 로드됐는지 확인!
5. 데이터 수집 셀 실행
6. 결과 확인 셀 실행 (df.head())
7. 저장 셀 실행
8. driver.quit() 실행 (반드시!)
```

### 결과 검증 체크리스트

```
□ df.shape  → 행/열 수가 예상과 맞는가?
□ df.head() → 첫 몇 개 데이터가 실제 사이트와 같은가?
□ df.isnull().sum() → None/NaN이 너무 많지 않은가?
□ df['가격'].dtype  → 숫자가 숫자형인가? 문자형으로 들어오진 않았나?
□ 실제 사이트에서 5개 골라서 직접 비교해보기
```

In [ ]:
# Step 1: 설치 확인 — 항상 제일 먼저 실행
# (AI가 준 코드 맨 위에 이 셀을 추가하세요)

import subprocess, sys

def check_package(pkg_name):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'show', pkg_name],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        for line in result.stdout.split('\n'):
            if line.startswith('Version'):
                print(f"✅ {pkg_name}: {line}")
    else:
        print(f"❌ {pkg_name}: 설치 안 됨 → pip install {pkg_name} 실행하세요")

check_package('selenium')
check_package('webdriver-manager')

print()
print("설치 안 된 것이 있으면 아래 셀의 주석을 해제하고 실행하세요.")

In [ ]:
# 필요할 때만 실행 (설치가 안 된 경우)
# !pip install selenium webdriver-manager

In [ ]:
# Step 2: import — AI가 준 코드에 항상 포함되는 "주문"
# 외울 필요 없어요. 이 블록을 통째로 복붙해서 쓰면 됩니다.

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

print("✅ import 완료!")

In [ ]:
# Step 3: 브라우저 열기
# AI가 준 옵션 코드가 이렇게 생겼으면 정상입니다.

options = webdriver.ChromeOptions()
# options.add_argument('--headless')  # 창 없이 실행 (서버용, 지금은 주석)
options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')
options.add_argument('--window-size=1280,900')
options.add_argument(
    'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'
)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
wait = WebDriverWait(driver, 10)

print("✅ Chrome 열림!")
print("   이제 수집 셀들을 실행하고, 마지막에 반드시 driver.quit() 을 실행하세요.")

In [ ]:
# Step 4-6: AI가 준 코드 실행 예시
# (quotes.toscrape.com/js/ — JS 렌더링 버전)

driver.get('http://quotes.toscrape.com/js/')

# 데이터 로드 대기
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.quote')))

quotes_el = driver.find_elements(By.CSS_SELECTOR, '.quote')
print(f"발견된 명언: {len(quotes_el)}개")

rows = []
for q in quotes_el:
    text   = q.find_element(By.CSS_SELECTOR, '.text').text
    author = q.find_element(By.CSS_SELECTOR, '.author').text
    tags   = [t.text for t in q.find_elements(By.CSS_SELECTOR, '.tag')]
    rows.append({'명언': text, '저자': author, '태그': ', '.join(tags)})

df_quotes = pd.DataFrame(rows)

# ─── 결과 검증 ──────────────────────────────────────────────
print()
print("=== 결과 검증 ===")
print(f"df.shape: {df_quotes.shape}")
print(f"결측치:   {df_quotes.isnull().sum().sum()}개")
print()
print(df_quotes.head(3).to_string(index=False))

In [ ]:
# Step 8: 브라우저 종료 — 반드시!
# 이 셀을 실행하지 않으면 Chrome 프로세스가 백그라운드에 남아요.

driver.quit()
print("✅ 브라우저 종료 완료")

---
## Part 6. 에러가 났을 때 AI에게 디버깅 요청하는 법

### 에러 디버깅 프롬프트 템플릿

에러가 나면 이렇게 AI에게 물어보세요:

```
아래 Selenium 코드를 실행했는데 에러가 납니다.

[에러 메시지]
(에러 메시지 전체 붙여넣기)

[코드]
(에러가 난 코드 붙여넣기)

[상황]
- 사이트: [URL]
- 어느 셀에서 에러가 났는지: [설명]
- Chrome 창은 열렸는가: [열림/안 열림]
- 데이터가 화면에는 보이는가: [보임/안 보임]

원인을 분석하고 수정된 코드를 줘.
```

---

### 자주 나는 에러와 원인

| 에러 | 원인 | AI에게 물어볼 때 |
|------|------|------------------|
| `NoSuchElementException` | 요소를 못 찾음 | "NoSuchElementException 에러인데 CSS 선택자가 잘못된 건지, 아직 로드 안 된 건지 알려줘" |
| `TimeoutException` | 10초 안에 요소 안 나타남 | "TimeoutException이 나는데 대기 시간을 늘려야 하는지, 선택자가 틀린 건지 알려줘" |
| `ElementClickInterceptedException` | 팝업이 버튼을 가리고 있음 | "버튼 클릭 시 ElementClickInterceptedException이 나는데 팝업 처리를 추가해줘" |
| `StaleElementReferenceException` | 페이지 바뀌고 요소가 사라짐 | "StaleElementReferenceException이 나는데 요소를 다시 찾는 방식으로 수정해줘" |
| 데이터가 0개 | JS 아직 실행 전 | "wait.until 조건을 어떻게 바꿔야 데이터가 로드된 후에 수집할 수 있는지 알려줘" |

---

### 선택자 수정 요청 프롬프트

```
코드를 실행했는데 데이터가 0개가 나옵니다.

[현재 코드의 선택자]
driver.find_elements(By.CSS_SELECTOR, '.product-item')

[사이트 URL]
[URL]

아래는 F12 개발자도구에서 확인한 HTML 일부입니다:
(HTML 코드 붙여넣기)

올바른 CSS 선택자를 찾아서 코드를 수정해줘.
```

> 💡 **Tip**: F12 → Elements 탭에서 원하는 요소를 우클릭 → Copy → Copy outerHTML
> 로 HTML을 복사해서 AI에게 붙여넣으면 선택자를 바로 고쳐줄 수 있어요.

---
## Part 7. 실전 사례: 무신사 스토어

무신사 스토어(`https://www.musinsa.com/main/musinsa/recommend?gf=A`)는
React 기반 동적 사이트입니다.
Ctrl+U 소스에는 상품 데이터가 없어요.

### 무신사용 AI 프롬프트 (복붙해서 바로 사용 가능)

```
무신사 스토어 추천 상품 페이지
(https://www.musinsa.com/main/musinsa/recommend?gf=A)
에서 상품 목록을 Selenium으로 크롤링하는 Python 코드를 작성해줘.

[수집할 항목]
- 상품명
- 브랜드명  
- 원가
- 할인가
- 할인율 (있는 경우)

[특이사항]
- 페이지 진입 시 팝업이 뜰 수 있음 (있으면 닫기 처리)
- 상품이 스크롤 시 로드될 수 있음 (스크롤 처리 포함)
- 로그인은 하지 않음

[기술 조건]
- selenium + webdriver-manager 사용
- ChromeOptions() 기본 설정 (disable-gpu, no-sandbox, window-size=1280,900, user-agent)
- WebDriverWait(driver, 15) 사용 (time.sleep은 스크롤 후에만)
- try-finally로 driver.quit() 보장
- 결과를 pandas DataFrame으로 저장
- CSV 저장 시 encoding='utf-8-sig'
- 수집 진행 상황을 print로 출력
- 주석은 한국어로 상세히
- Jupyter Notebook 셀 단위로 구분해줘
```

---

### 코드를 받은 후 실행 순서

```
1. 설치 확인 셀 실행
2. import 셀 실행
3. 브라우저 열기 셀 실행 → Chrome 창이 뜸
4. 브라우저 창을 보면서 페이지가 제대로 로드되는지 확인
   → 상품이 보이면 OK, 빈 화면이면 AI에게 로딩 대기 코드 수정 요청
5. 수집 셀 실행
6. df.head()로 결과 확인 → 실제 사이트 상품과 비교
7. CSV 저장
8. driver.quit()
```

In [ ]:
# ══════════════════════════════════════════════════════════════
# 아래는 AI가 줄 코드의 예시 구조입니다.
# 실제 선택자는 사이트 구조에 따라 다르므로,
# 위의 프롬프트로 AI에게 요청해서 받아 사용하세요.
# ══════════════════════════════════════════════════════════════

MUSINSA_URL = 'https://www.musinsa.com/main/musinsa/recommend?gf=A'

print("AI에게 아래 프롬프트로 코드를 요청하세요:")
print()
print("""
무신사 스토어 추천 상품 페이지
(https://www.musinsa.com/main/musinsa/recommend?gf=A)
에서 상품 목록을 Selenium으로 크롤링하는 Python 코드를 작성해줘.

[수집할 항목]
- 상품명, 브랜드명, 원가, 할인가, 할인율

[기술 조건]
- selenium + webdriver-manager
- ChromeOptions() 기본 설정
- WebDriverWait(driver, 15) 사용
- 팝업 처리 포함 (try-except)
- 스크롤 처리 포함
- try-finally로 driver.quit() 보장
- DataFrame → CSV (utf-8-sig)
- 주석 한국어, Jupyter 셀 구분
""")

---
## 참고: AI가 주는 Selenium 코드의 표준 구조

AI가 준 코드가 아래 구조를 따르는지 확인하세요.
이 구조에서 벗어나면 AI에게 수정 요청하세요.

```python
# ── 셀 1: import ──────────────────────────────────
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time, pandas as pd

# ── 셀 2: 브라우저 열기 ───────────────────────────
options = webdriver.ChromeOptions()
options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')
options.add_argument('--window-size=1280,900')
options.add_argument('user-agent=Mozilla/5.0 ...')

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
wait = WebDriverWait(driver, 10)

# ── 셀 3: 수집 ────────────────────────────────────
try:
    driver.get('사이트 URL')
    
    # 팝업 처리
    try:
        close_btn = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, '팝업닫기선택자')
        ))
        close_btn.click()
    except:
        pass  # 팝업 없으면 그냥 넘어감
    
    # 데이터 로드 대기
    wait.until(EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, '아이템선택자')
    ))
    
    # 데이터 수집
    items = driver.find_elements(By.CSS_SELECTOR, '아이템선택자')
    rows = []
    for item in items:
        name = item.find_element(By.CSS_SELECTOR, '이름선택자').text
        rows.append({'이름': name})
    
    df = pd.DataFrame(rows)
    df.to_csv('결과.csv', index=False, encoding='utf-8-sig')
    print(f'완료: {len(df)}개')

finally:
    driver.quit()  # ← try-finally로 항상 실행!
```

---

### Selenium 핵심 코드 3줄 (이것만 알면 AI 코드 읽기 가능)

```python
driver.find_elements(By.CSS_SELECTOR, '.item')  # 여러 개 찾기
element.find_element(By.CSS_SELECTOR, '.name')  # 하나 찾기 (요소 안에서)
element.text                                     # 텍스트 가져오기
element.get_attribute('href')                    # 속성값 가져오기
```

---
## ✅ 핵심 정리

### Selenium × AI 활용 5단계

```
Step 1: Ctrl+U로 동적 사이트 확인
         → 데이터가 소스에 없으면 Selenium 필요

Step 2: 템플릿에서 상황에 맞는 것 골라서
         [대괄호] 부분만 채워서 AI에게 요청

Step 3: AI가 준 코드를 순서대로 실행
         → 브라우저 창을 보면서 제대로 동작하는지 확인

Step 4: df.head() + df.shape으로 결과 검증
         → 실제 사이트와 비교

Step 5: 에러 나면 에러 메시지 + 코드를 그대로 AI에게
         → "원인 분석 + 수정된 코드 줘"
```

---

### BeautifulSoup vs Selenium 선택 기준

| 상황 | 도구 |
|------|------|
| Ctrl+U 소스에 데이터 있음 | BeautifulSoup (빠름, 안정적) |
| JS 렌더링, 소스에 데이터 없음 | Selenium |
| 무한스크롤 | Selenium |
| 버튼 클릭, 필터 적용 | Selenium |
| 로그인 후 수집 | Selenium |
| 수천 페이지 대량 수집 | BeautifulSoup (속도 이점) |

---

> **다음 노트북**: Open API 데이터 수집
> → 크롤링 없이 공식 데이터를 직접 받는 법 (공공데이터포털, 기상청 등)